In [1]:
import torch
import torch.nn as nn

from torchvision.models import (
    resnet18,
    ResNet18_Weights
)

encoder = resnet18(
    weights=ResNet18_Weights.DEFAULT
)

print(encoder)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\User/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [01:15<00:00, 624kB/s] 


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [2]:
original_weights = encoder.conv1.weight.detach().clone()

print("Original weight shape:")
print(original_weights.shape)

new_conv = nn.Conv2d(
    in_channels=2,
    out_channels=64,
    kernel_size=7,
    stride=2,
    padding=3,
    bias=False
)

with torch.no_grad():

    adapted_weights = original_weights.mean(
        dim=1,
        keepdim=True
    )

    adapted_weights = adapted_weights.repeat(
        1, 2, 1, 1
    )

    adapted_weights = adapted_weights * (3 / 2)

    new_conv.weight.copy_(adapted_weights)

encoder.conv1 = new_conv

print("\nAdapted convolution:")
print(encoder.conv1)

print("\nAdapted weight shape:")
print(encoder.conv1.weight.shape)

Original weight shape:
torch.Size([64, 3, 7, 7])

Adapted convolution:
Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)

Adapted weight shape:
torch.Size([64, 2, 7, 7])


In [3]:
encoder.eval()

x = torch.randn(1, 2, 512, 512)

with torch.inference_mode():

    e0 = encoder.relu(
        encoder.bn1(
            encoder.conv1(x)
        )
    )

    e1 = encoder.layer1(
        encoder.maxpool(e0)
    )

    e2 = encoder.layer2(e1)

    e3 = encoder.layer3(e2)

    e4 = encoder.layer4(e3)

print("Input:", x.shape)
print("Encoder 0:", e0.shape)
print("Encoder 1:", e1.shape)
print("Encoder 2:", e2.shape)
print("Encoder 3:", e3.shape)
print("Encoder 4:", e4.shape)

Input: torch.Size([1, 2, 512, 512])
Encoder 0: torch.Size([1, 64, 256, 256])
Encoder 1: torch.Size([1, 64, 128, 128])
Encoder 2: torch.Size([1, 128, 64, 64])
Encoder 3: torch.Size([1, 256, 32, 32])
Encoder 4: torch.Size([1, 512, 16, 16])


In [4]:
from floodvision.models.resnet_unet import ResNetUNet

model = ResNetUNet(
    in_channels=2,
    out_channels=1,
    pretrained=True
)

print(model)

ResNetUNet(
  (encoder0): Sequential(
    (0): Conv2d(2, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (pool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (encoder1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True

In [5]:
from floodvision.models.baseline_cnn import BaselineCNN
from floodvision.models.unet import UNet

models = {
    "Baseline CNN": BaselineCNN(),
    "U-Net": UNet(),
    "ResNet18 U-Net": model
}

for name, net in models.items():

    total = sum(
        p.numel()
        for p in net.parameters()
    )

    trainable = sum(
        p.numel()
        for p in net.parameters()
        if p.requires_grad
    )

    print(f"\n{name}")
    print(f"Total parameters: {total:,}")
    print(f"Trainable parameters: {trainable:,}")
    print(f"FP32 weights: {total * 4 / 1024**2:.2f} MiB")


Baseline CNN
Total parameters: 434,209
Trainable parameters: 434,209
FP32 weights: 1.66 MiB

U-Net
Total parameters: 482,593
Trainable parameters: 482,593
FP32 weights: 1.84 MiB

ResNet18 U-Net
Total parameters: 14,238,145
Trainable parameters: 14,238,145
FP32 weights: 54.31 MiB


In [6]:
model.eval()

x = torch.randn(1, 2, 512, 512)

with torch.inference_mode():
    output = model(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Output dtype:", output.dtype)
print("Minimum logit:", output.min().item())
print("Maximum logit:", output.max().item())

assert output.shape == (1, 1, 512, 512)
assert torch.isfinite(output).all().item()

print("\nForward pass successful!")

Input shape: torch.Size([1, 2, 512, 512])
Output shape: torch.Size([1, 1, 512, 512])
Output dtype: torch.float32
Minimum logit: -0.022619154304265976
Maximum logit: -0.007238588761538267

Forward pass successful!


In [7]:
from floodvision.losses.combined import BCEDiceLoss

model.train()
model.zero_grad(set_to_none=True)

x = torch.randn(2, 2, 128, 128)

target = torch.randint(
    0, 2, (2, 1, 128, 128)
).float()

# Simulate invalid pixels
target[:, :, :10, :10] = -1

criterion = BCEDiceLoss(
    bce_weight=0.5,
    dice_weight=0.5,
    pos_weight=None
)

logits = model(x)

loss = criterion(logits, target)

assert torch.isfinite(loss).item()

loss.backward()

print("Loss:", loss.item())
print("Backward pass successful!")

Loss: 0.5951520204544067
Backward pass successful!


In [8]:
missing_gradients = []
invalid_gradients = []
nonzero_gradients = []

for name, param in model.named_parameters():

    if not param.requires_grad:
        continue

    if param.grad is None:
        missing_gradients.append(name)

    elif not torch.isfinite(param.grad).all():
        invalid_gradients.append(name)

    elif torch.count_nonzero(param.grad).item() > 0:
        nonzero_gradients.append(name)

print("Missing gradients:", missing_gradients)
print("Invalid gradients:", invalid_gradients)

print(
    "Non-zero gradient tensors:",
    len(nonzero_gradients)
)

assert len(missing_gradients) == 0
assert len(invalid_gradients) == 0
assert len(nonzero_gradients) > 0

print("\nGradient validation successful!")

Missing gradients: []
Invalid gradients: []
Non-zero gradient tensors: 102

Gradient validation successful!


In [9]:
print(
    "Encoder gradient:",
    model.encoder0[0].weight.grad.norm().item()
)

print(
    "Decoder gradient:",
    model.decoder3.up.weight.grad.norm().item()
)

print(
    "Output gradient:",
    model.output.weight.grad.norm().item()
)

Encoder gradient: 0.2109743356704712
Decoder gradient: 0.03202351927757263
Output gradient: 0.05322188884019852


In [10]:
from floodvision.models.factory import (
    build_model,
    MODEL_REGISTRY
)

print("Available models:", list(MODEL_REGISTRY))

factory_model = build_model({
    "name": "resnet_unet",
    "in_channels": 2,
    "out_channels": 1,
    "pretrained": False
})

assert isinstance(factory_model, ResNetUNet)

print("Model:", type(factory_model).__name__)

print(
    "Parameters:",
    sum(p.numel() for p in factory_model.parameters())
)

print("Factory validation successful!")

Available models: ['baseline_cnn', 'unet', 'resnet_unet']
Model: ResNetUNet
Parameters: 14238145
Factory validation successful!
